In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from datetime import datetime
import time

import warnings
warnings.filterwarnings('ignore')

import sklearn
from sklearn import metrics
from sklearn.metrics import confusion_matrix, f1_score

import random, os, json
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Masking, Dropout, Dense, Dropout, Flatten, Conv1D
from tensorflow.keras import backend as K
from tensorflow.keras import layers
from tensorflow.keras import Input
from tensorflow.keras import optimizers
from tensorflow.keras.models import Model

from joblib import Parallel, delayed
import multiprocessing

import pickle

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from joblib import Parallel, delayed
import multiprocessing
from tensorflow.keras import regularizers
from sklearn.model_selection import KFold


In [ ]:
def reset_keras(seed=42):
    """Function to ensure that results from Keras models
    are consistent and reproducible across different runs"""
    
    K.clear_session()
    # 1. Set `PYTHONHASHSEED` environment variable at a fixed value
    os.environ['PYTHONHASHSEED']=str(seed)
    # 2. Set `python` built-in pseudo-random generator at a fixed value
    random.seed(seed)
    # 3. Set `numpy` pseudo-random generator at a fixed value
    np.random.seed(seed)
    # 4. Set `tensorflow` pseudo-random generator at a fixed value
    tf.random.set_seed(seed)

In [ ]:
def build_model(hyperparameters):
    """
    Builds a Transformer model based on the provided training data and hyperparameters.
    Args:
        - hyperparameters: Dictionary containing hyperparameters.
    Returns:
        - model: A tf.keras.Model with the compiled model.
    """
    dropout = hyperparameters["dropout"]
    num_heads = hyperparameters["num_heads"]
    num_transformer_blocks = hyperparameters["num_transformer_blocks"]
    activation = hyperparameters['activation']
    l2_reg = hyperparameters.get('regularizer', {}).get('value', 0.0)
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=hyperparameters["lr_scheduler"])


    input = Input(shape=(hyperparameters["n_time_steps"], hyperparameters["layer_list"][0]))
    x = input
    masked = x

    for _ in range(num_transformer_blocks):
        # NORMALIZATION AND ATTENTION
        x_norm = layers.LayerNormalization()(masked)
        x_att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=input.shape[-1])(x_norm, x_norm)
        x_att_drop = layers.Dropout(dropout)(x_att)
        res = x_att_drop + masked

        # FEED FORWARD PART
        x_ffn_norm = layers.LayerNormalization()(res)
        x_ffn = layers.Dense(
            input.shape[-1], activation=activation, kernel_regularizer=regularizers.l2(l2_reg)
        )(x_ffn_norm)
        x = x_ffn + res

    x = layers.Dropout(dropout)(x)
    x = layers.Dense(
        hyperparameters["layer_list"][1], activation=activation, kernel_regularizer=regularizers.l2(l2_reg)
    )(x)
    x = layers.Dropout(dropout)(x)

    output = layers.GlobalMaxPooling1D()(x)  
    output = layers.Dense(1, activation='sigmoid',kernel_regularizer=regularizers.l2(l2_reg))(output)  # Output: (None, 1)

    model = Model(input, output)
    
    # COMPILE 
    model.compile(
        loss='binary_crossentropy',
        optimizer=optimizer,
        metrics=['accuracy', "AUC"], weighted_metrics = [] 
    )
    
    return model

def run_network(X_train, X_val, y_train, y_val, 
                hyperparameters, seed):
    """
    Trains and evaluates the built Transformer model based on the provided data and hyperparameters.
    Args:
        - X_train, X_val, y_train, y_val: numpy.ndarray. Training (T) and Validation (V) data labels.
        - sample_weights_train, sample_weights_val: numpy.ndarray. Weights for the T and V data to handle class imbalance.
        - hyperparameters: Dictionary containing the hyperparameters.
        - seed: Integer seed for reproducibility.
    Returns:
        - model: A tf.keras.Model with the trained model
        - hist:  The training history
        - earlystopping: The early stopping callback.
    """
    
    # Build the transformer
    model = None
    model = build_model(hyperparameters) 

    earlystopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        min_delta=hyperparameters["mindelta"],
        patience=hyperparameters["patience"],
        restore_best_weights=True,
        mode="min",
    )
    
    X_val = tf.cast(X_val, tf.float64)
    y_val = tf.cast(y_val, tf.float64)
    X_train = tf.cast(X_train, tf.float64)
    y_train = tf.cast(y_train, tf.float64)

    start_time = time.time()

    hist = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        callbacks=[earlystopping],
        batch_size=hyperparameters['batch_size'],
        epochs=hyperparameters['n_epochs_max'],
        verbose=hyperparameters["verbose"],
    )

    end_time = time.time()
    training_time = end_time - start_time

    return model, hist, training_time

In [ ]:
def evaluate_combination(k, l, m, n, a, b, c, hyperparameters, dropout, layer_list, lr_scheduler, regularizer, 
                         num_transformer_blocks, activation, regularization, num_heads, seed, split, kfold, n_time_steps):
    
    hyperparameters_copy = hyperparameters.copy()
    hyperparameters_copy['dropout'] = dropout[k]
    hyperparameters_copy['layer_list'] = layer_list[l]
    hyperparameters_copy['lr_scheduler'] = lr_scheduler[m]
    hyperparameters_copy['regularizer'] = regularizer[n]
    hyperparameters_copy['num_transformer_blocks'] = num_transformer_blocks[a]
    hyperparameters_copy['activation'] = activation[b]
    hyperparameters_copy['num_heads'] = num_heads[c]
    
    kfold = hyperparameters_copy['kfold'] = kfold
    v_val_loss = []

    # Carga del dataset completo
    X = np.load("../df_to_load/DataToPaperAndTFM_Mod1/Subconjuntos_3D/S" + str(split) + "/X_train_tensor.npy")
    y = pd.read_csv("../df_to_load/DataToPaperAndTFM_Mod1/Subconjuntos_3D/S" + str(split) + "/y_train_tensor.csv")[['MR']].MR.values

    # K-Fold
    kf = KFold(n_splits=kfold, shuffle=True, random_state=seed)

    for train_index, val_index in kf.split(X):
        X_train_fold, X_val_fold = X[train_index], X[val_index]
        y_train_fold, y_val_fold = y[train_index], y[val_index]

        reset_keras()

        model, hist, early = run_network(
            X_train_fold, X_val_fold,
            y_train_fold,
            y_val_fold,
            hyperparameters_copy,
            seed
        )

        v_val_loss.append(np.min(hist.history["val_loss"]))

    metric_dev = np.mean(v_val_loss)
    return (metric_dev, k, l, m, n, a, b, c, X_train_fold, y_train_fold, X_val_fold, y_val_fold)


def myCVGridParallel(hyperparameters, dropout, layer_list, lr_scheduler, regularizer, 
                    num_transformer_blocks, activation, regularization, num_heads, 
                     seed, split, kfold, n_time_steps):
    """Parallelized Grid Search. 
       Calculate metricDev based on the evaluation. Compares the metricDev with the current bestMetricDev. 
       If better, updates bestMetricDev and stores those hyperparameters in bestHyperparameters.
       
    Args:
        - hyperparameters: Dictionary containing the hyperparameters.
        - dropout: A list of dropout rates.
        - lr_scheduler: A list of learning rates.
        - layers: A list of layer configurations.
        - seed : Seed value for reproducibility.
        - split: String indicating the data split.
        - norm: String with the type of normalization applied to the data.
    Returns:
        - bestHyperparameters: A dictionary with the best hyperparameters found and Train and Val data.
    """
    bestHyperparameters = {}
    bestMetricDev = np.inf

    num_cores = 20
    results = Parallel(n_jobs=num_cores)(
        delayed(evaluate_combination)(k, l, m, n, a, b, c, hyperparameters, dropout, layer_list, lr_scheduler, 
                                      regularizer, num_transformer_blocks, activation, regularization, 
                                      num_heads, seed, split, kfold, n_time_steps)
        for k in range(len(dropout))
        for l in range(len(layer_list))
        for m in range(len(lr_scheduler))
        for n in range(len(regularizer))
        for a in range(len(num_transformer_blocks))
        for b in range(len(activation))
        for c in range(len(num_heads))
    )

    for metric_dev, k, l, m, n, a, b, c, X_train, y_train, X_val, y_val in results:
        if metric_dev < bestMetricDev:
            print("\t\t\tCambio the best", bestMetricDev, "por metric dev:", metric_dev)
            bestMetricDev = metric_dev
            bestHyperparameters = {
                'dropout': dropout[k],
                'layer_list': layer_list[l],
                'lr_scheduler': lr_scheduler[m],
                'regularizer': regularizer[n],
                'num_transformer_blocks': num_transformer_blocks[a],
                'activation': activation[b],
                'num_heads': num_heads[c],
                'X_train': X_train,
                'y_train': y_train,
                'X_val': X_val,
                'y_val': y_val
            }

    return bestHyperparameters

def calculate_metrics(model, X_test, y_test):

    y_pred_test = model.predict(X_test)
    
    # Calculate metrics
    accuracy_test = sklearn.metrics.accuracy_score(y_test, np.round(y_pred_test))
    tn, fp, fn, tp = confusion_matrix(y_test, np.round(y_pred_test)).ravel()
    specificity = tn / (tn + fp)
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1_score = (2 * recall * precision) / (recall + precision)
    roc_auc = sklearn.metrics.roc_auc_score(y_test, y_pred_test)
    
    # Dataframe
    df_metrics = pd.DataFrame({
        'accuracy': [accuracy_test],
        'specificity': [specificity],
        'recall': [recall],
        'precision': [precision],
        'f1_score': [f1_score],
        'roc_auc': [roc_auc]
    })
    return df_metrics

# HYPERPARAMETERS

- **seeds**: Seed values to ensure reproducibility.
- **input_shape**: Number of features in each time step of the input data.
- **n_time_steps**: Number of time steps in the input sequence.
- **batch_size**: Number of batches for training.
- **n_epochs_max**: Maximum number of epochs for training.
- **layer_list**: A list with different configurations for the layers of the model.
- **dropout**: Dropout rates.
- **lr_scheduler**: Learning rates.
- **norm**: Type of normalization applied to the data.
- **num_heads**: Number of attention heads in the multi-head attention mechanism.
- **num_transformer_blocks**: Number of transformer blocks.
- **epsilon**: Avoid zero division in the normalization layer.

In [ ]:
seeds = [9, 34, 76, 105, 227]

input_shape = 50
n_time_steps = 7
batch_size = 32
n_epochs_max = 100
kfold = 5
layer_list = [
    [input_shape, 20, 1],  [input_shape, 30, 1], [input_shape, 35, 1], 
    [input_shape, 40, 1], [input_shape, 45, 1], [input_shape, 50, 1]
]

dropout = [0, 0.1, 0.15, 0.2, 0.3]
lr_scheduler = [1e-3, 1e-4, 1e-5]

adjustment_factor = [1] 

activation = ['relu', 'LeakyReLU']

#Transformer---------------
num_heads = [2, 4, 6, 8]

num_transformer_blocks = [2, 4, 6]

regularizer = [
    {'type': 'l2', 'value': 0},
    {'type': 'l2', 'value': 1e-4},
    {'type': 'l2', 'value': 1e-2}
]

#---------------------------

hyperparameters = {
    "n_time_steps": n_time_steps,
    "mask_value": 0.,
    "kfold": kfold,
    "batch_size": batch_size,
    "n_epochs_max": n_epochs_max,
    "monitor": "val_loss",
    "mindelta": 0,
    "patience": 10,
    "dropout": 0.0,
    "verbose": 0,
    "input_shape": input_shape,
    "num_heads": num_heads,
    "num_transformer_blocks": 0,
    "epsilon": 0
}

# RUNNING AND TRYING ON TEST

In [ ]:
import os
import time
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

def load_from_pickle(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)
    
run_model = True
if run_model:
    loss_train = []
    loss_dev = []
    v_models = []
    v_accuracy_test = []
    v_specificity = []
    v_precision = []
    v_recall = []
    v_f1score = []
    v_roc = []

    bestHyperparameters_bySplit = {}
    y_pred_by_split = {}
    metrics_splits = []

    for i in [1, 2, 3, 4, 5]:
        init = time.time()
        X_test = np.load("../df_to_load/DataToPaperAndTFM_Mod1/Subconjuntos_3D/S" +str(i)+ "/X_test_tensor"+".npy")
        y_test = pd.read_csv("../df_to_load/DataToPaperAndTFM_Mod1/Subconjuntos_3D/S" +str(i) + "/y_test_tensor"+".csv")[['MR']].MR.values
        
        # GridSearch of hyperparameters 
        bestHyperparameters = myCVGridParallel(
            hyperparameters,
            dropout,
            layer_list,
            lr_scheduler,
            regularizer,
            num_transformer_blocks,
            activation,
            regularizer, 
            num_heads,
            seeds[i-1],
            f"{i}",
            kfold,
            n_time_steps
        )
        fin = time.time()
        X_train = bestHyperparameters["X_train"]
        y_train = bestHyperparameters["y_train"]
        X_val = bestHyperparameters["X_val"]
        y_val = bestHyperparameters["y_val"]

        # Save best hyperparameters for current split
        split_directory = f'./Results_Transformer/split_{i}'
        if not os.path.exists(split_directory):
            os.makedirs(split_directory)

        with open(os.path.join(split_directory, f"bestHyperparameters_split_{i}.pkl"), 'wb') as f:
            pickle.dump(bestHyperparameters, f)
                    
        hyperparameters.update({
            "dropout": bestHyperparameters["dropout"],
            "layer_list": bestHyperparameters["layer_list"],
            "lr_scheduler": bestHyperparameters["lr_scheduler"], 
            "regularizer": bestHyperparameters["regularizer"],
            "num_transformer_blocks": bestHyperparameters["num_transformer_blocks"],
            "activation": bestHyperparameters["activation"],
            "num_heads": bestHyperparameters["num_heads"],
        })


        #--- TRY ON TEST -----------------------------------------------------------------------#

        reset_keras()

        model, hist, early = run_network(
            X_train, 
            X_val,
            y_train, 
            y_val,
            hyperparameters,
            seeds[i-1]
        )

        v_models.append(model)
        loss_train.append(hist.history['loss'])
        loss_dev.append(hist.history['val_loss'])

        y_pred = model.predict(X_test)
        y_pred_by_split[str(i)] = y_pred
        
        accuracy_test = sklearn.metrics.accuracy_score(y_test, np.round(y_pred))
        tn, fp, fn, tp = confusion_matrix(y_test, np.round(y_pred)).ravel()
        roc = sklearn.metrics.roc_auc_score(y_test, y_pred)


        v_accuracy_test.append(accuracy_test)
        v_specificity.append(tn / (tn + fp))
        v_precision.append(tp / (tp + fp))
        v_recall.append(tp / (tp + fn))
        v_f1score.append((2 * v_recall[i-1] * v_precision[i-1]) / (v_recall[i-1] + v_precision[i-1]))
        v_roc.append(roc)
        
        # Save y_pred and metrics for current split
        with open(os.path.join(split_directory, f"y_pred_split_{i}.pkl"), 'wb') as f:
            pickle.dump(y_pred, f)
        with open(os.path.join(split_directory, f"metrics_{i}.pkl"), 'wb') as f:
            pickle.dump(metrics, f)
            

    # END EXECUTION - SAVE AGGREGATED RESULTS
    directory = './Results_Transformer'
    if not os.path.exists(directory):
        os.makedirs(directory)

In [ ]:
all_metrics = {
    "roc_auc": v_roc,
    "sensitivity": v_recall,
    "specificity": v_specificity
}

result = {}

for key, values in all_metrics.items():
    mean_value = np.mean(values)
    std_value = np.std(values)
    result[key] = {
        "mean": mean_value,
        "std": std_value
    }

with open("./Results_Transformer/summary.json", "w") as f:
    json.dump(result, f, indent=4)

print(result)


for i in range(5):
    split_data = {
        "metrics": {
            "roc_auc": all_metrics["roc_auc"][i],
            "sensitivity": all_metrics["sensitivity"][i],
            "specificity": all_metrics["specificity"][i]
        }
    }

    split_folder = f"./Results_Transformer/split_{i+1}"
    os.makedirs(split_folder, exist_ok=True)

    with open(f"{split_folder}/results.json", "w") as f:
        json.dump(split_data, f, indent=4)

    print(f"Saved: {split_folder}/results.json")